# 第53章 聚类热力图（clustermap）

用层次聚类重新排列矩阵，发现相似行列和潜在群组。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

行列数量较多，希望按相似模式自动分组。

## 数据结构

行和列均为可比较的数值矩阵；通常需要标准化。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 z_score=1 改为 standard_scale=0，对比按列标准化与按行标准化的聚类结果
2. 修改 row_cluster=True 为 row_cluster=False，观察禁用行聚类对树状图的影响
3. 调整 method 参数（如 'average' 或 'complete'），说明不同连接方法对聚类结构的影响


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook")
from js import window
base_url = window.location.origin
diamonds = pd.read_csv(f"{base_url}/datasets/diamonds.csv")
orders_full = diamonds.assign(
    category=diamonds["cut"], channel=diamonds["color"], region=diamonds["clarity"],
    order_value=diamonds["price"], items=diamonds["carat"],
    satisfied=np.where(diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"),
)
orders = orders_full.sample(2_000, random_state=36).copy()
taxis = pd.read_csv(f"{base_url}/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"), visits=taxis["distance"],
    ad_spend=taxis["tip"], sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(min(2_000, len(marketing_full)), random_state=36).copy()
flights = pd.read_csv(f"{base_url}/datasets/flights.csv")
daily = flights.assign(
    date=pd.to_datetime(flights["year"].astype("string") + "-" + flights["month"] + "-01"),
    region="AirPassengers", sales=flights["passengers"],
)
print(f"Diamonds：{len(diamonds):,} 行；NYC Taxis：{len(taxis):,} 行；Flights：{len(flights):,} 行")
print("图表兼容列均由公开数据原始字段直接映射；高成本图使用固定 2,000 行样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import scipy

category_region = orders.pivot_table(index="category", columns="region", values="order_value", aggfunc="mean")
grid = sns.clustermap(category_region, cmap="Blues", annot=True, fmt=".0f", figsize=(7, 6), row_cluster=True, col_cluster=True)
grid.fig.suptitle("品类与区域客单价聚类", y=1.02)
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
profile = marketing.groupby("channel")[["visits", "ad_spend", "sales", "conversion"]].mean()
grid = sns.clustermap(profile, z_score=1, cmap="vlag", center=0, annot=True, fmt=".2f", figsize=(8, 6))
grid.fig.suptitle("渠道指标标准化聚类", y=1.02)
plt.show()


## 3. 参数说明

- z_score/standard_scale：标准化
- method：连接方法
- metric：距离
- row_cluster/col_cluster：聚类方向


## 4. 结果解读

树状图表达合并顺序和距离，色块表达标准化后的相对模式。


## 常见误区

- 量纲不同却不标准化
- 把聚类结果当作唯一真实分类
- 样本太少或缺失太多


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
region_profile = orders.groupby("region").agg(order_value=("order_value", "mean"), items=("items", "mean"))
grid = sns.clustermap(region_profile, standard_scale=1, cmap="YlGnBu", annot=True, fmt=".2f", figsize=(7, 5))
grid.fig.suptitle("区域订单特征聚类", y=1.02)
plt.show()


## 本章小结

用层次聚类重新排列矩阵，发现相似行列和潜在群组。


### 你已经掌握

- 判断聚类热力图（clustermap）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 行列数量较多，希望按相似模式自动分组。 |
| 数据结构 | 行和列均为可比较的数值矩阵；通常需要标准化。 |
| 结果解读 | 树状图表达合并顺序和距离，色块表达标准化后的相对模式。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `z_score/standard_scale` | 标准化 |
| `method` | 连接方法 |
| `metric` | 距离 |
| `row_cluster/col_cluster` | 聚类方向 |


### 需要注意

- 量纲不同却不标准化
- 把聚类结果当作唯一真实分类
- 样本太少或缺失太多


### 完成检查

- [ ] 能判断什么问题适合使用聚类热力图（clustermap）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
